# A Coffee Shop Daily Profit Forecasting Model

In the **Monte Carlo Method Article** we saw, **what the Monte Carlo Method is**, **how it works**, and **what it needs to be working**!

So, that was the required knowledge we should have gathered, in order to create the algorithm of this **forecasting model**.And now, when we have those skills, we can easily start the development.

##### But first, let's actually, understand and specify the aim of the **forecasting model** we are going to develop:

Take for example a local coffee shop.The owner, tries to **predict** the daily income of it's coffee, but like many others, he relies on **deterministic, statical** assumptions, which creates a **false** sense of security, masking the very real risks of sudden supplier price hikes, **unexpected** bad weather, or seasonal dips in foot traffic.

The fundamental flaw in relying on a single-point estimations for future business outcomes, is that it completely ignores the **inherent volatility** of the real world.When a coffee shop owner assumes they will serve exactly 150 customers every day, or that the wholesale cost of coffee beans will remain perfectly static, they are building a financial model on a **fragile foundation**. 

As it is **expected** the methods, used by the owner are false every single time, just because, even that he tries to predict some edge case scenarios, there will always be something unexpected that is simply beyond his powers to foresee and predict.

However, the owner has finaly decided that he wants to have a better clarity on his financial incomes.In order to so, the owner needs a solid forecastig model, which predicts his coffee shop daily profit.

#### And here we come to the aim of the project:
> To overcome the **limitations** of the **traditional forecasting**, this project implements a **Monte Carlo Risk Simulation**. Rather than relying on **rigid, single-point assumptions**, this **computational framework** embraces real-world uncertainty by assigning **statistical probability distributions** to volatile variables, such as daily customer foot traffic and fluctuating wholesale costs.

By running **thousands** of simulated business days, the engine generates a **comprehensive** spectrum of possible financial outcomes and calculates the exact **likelihood** of each occurring.

For the coffee shop owner, this means replacing a fragile 'best guess' with a robust, data-driven forecast. It provides immediate clarity on critical business questions, highlighting not just the **expected average profit**, but the **exact probability** of operating at a **loss** on any given day or the **likelihood** of achieving specific revenue targets.

### With this clarification, we are totally ready to start the development of the algorithm and all of it's components!

Ok now let's first start by defining the ***Base Mathematical Model***, on which the daily profit is going to be calculated.

The **daily profit** is defined as:
$$
    Profit = (N \times (P - C_v)) - C_f
$$

Where:
- $N$: Number of customers
- $P$: Price per order
- $C_v$: Variable cost per order
- $C_f$: Fixed daily cost(overhead)

Ok, we have defined the base model, and identified it's components, but which of these components are actually going to be **predicted**?

As *Chat gpt*[1] says: \
In real-world systems, uncertainty arises from:
- Customer arrivals (demand uncertainty)
- Supply cost variability
- Market conditions

And this is logical, because we can never be sure the amount of customers, which are going to come and buy coffee. \
We also can't be sure about the variable cost per order, such us beans, milk, cups - these are all defined as variables with unsetteled prices and inflationary changes!

Now we can surely say that the **Random Varibles**, in this case are going to be:
- $N$: The number of customers
    - Daily variability (weekday vs weekend)
    - Primary driver of revenue
    - Major contributor to the variance in profit

- $C_v$: The variable cost per order
    - Supply chain volatility (coffee beans, milk)
    - Market price changes (commodities)
    - Operational inefficiencies
    - Supplier variability

However, we should also define the **Deterministic (Fixed) Variables**:
- $P$: The price per Order
    - Set by the business
    - Changes infrequently relative to daily operations

- $C_f$: The daily overhead
    - Includes rent, salaries, utilities
    - Typically constant on a daily basis
    - Known in advance with high certainty

Okay, now that we've clarified each of the model components and understood their roles in the algorithm,let's also put a little attention on the thing that is going to drive the algorithm and it's simulations:
#### - The Probability distributions

We drive the simulations, by sampling many times from the distributions of the random variables.

Each simulation run represents:
> A possible prediction of a business day.

Thus, the entire variability of profit emerges from:
- Demand randomness (N)
- Cost randomness ($C_v$)

### Which distributions are going to be used and why?
By making a comprehensive research I have stopped at the [*Normal*](https://en.wikipedia.org/wiki/Normal_distribution) as a distribution for the **customers** and the [*Triangular*](https://en.wikipedia.org/wiki/Triangular_distribution) for the variable costs.

#### So, why them?
### 1.Normal Distribution for Customer Count ($N$):
$$
    N \sim \mathcal{N}(\mu_N, \sigma_{N}^{2})
$$
As the **Central Limit Theorem** states:
> The sum of many independent random influences converges to a Normal distribution.

If we assume that customer arrivals are influenced by many small, independent factors:
- Individual decisions
- Weather
- habits
- local activity

Then:
$$
    N = X_1 + X_2 + X_3,...X_i
$$

Where each $X_i$ is a small contributing factor.

So, we can say that $N$ is approximately Normal.

### 2.Triangular Distribution for Variable Cost ($C_v$):
$$
    C_v \sim \mathcal{T}(a, m, b)
$$

Where:
- a: minimum
- m: most likely (mode)
- b: maximum

For a coffee shop:
- Minimum cost = stable supply scenario
- Maximum cost = shortage/spike
- Mode = typical operating condition

Why not **Normal** for the variable costs?
- Costs are bounded (cannot be negative)
- Often asymmetric (price spikes, drops)
- Normal distribution violates these constraints

Well, I think we've gone through all the important details that we had to take into account and specified everything that needed to be specified, and it's time to move on to the essential part, namely the development of the algorithm.

To start with, we are going to implement, a general version of the algorithm.By general, I mean that it will be used for the calculation of the main, but still very important, statistics such as, average profit, probability of loss, probability of achiving specific profit prices etc.

Let's make a quick structure of the algorithm and list down all of it's componenets:
- Random Variable Generators:
    - Responsible for sampling $N$ and $C_v$

- Profit Evaluation:
    - Implements: $ (N \times (P - C_v)) - C_f $

- Monte Carlo Simulations Engine:
    - Combines everything
    - Drives the simulations
    - Returns the calculated profits

- Input Parameter Module
    - Distribution parameters
    - Simulation size
    - Variable (fixed) Constants

- Extracting the results:
    - Using the input data, which drives the Monte Carlo simulations and gets the calculated profits
    - Returning statical data


Now the implementation:

In [34]:
import numpy as np

First let's craete the function that will sample the customets from the **Normal distribution**:

In [2]:
def sample_customers_normal(mean, sd, size):
    """
    1) Samples the customers from a Normal distribution, by ensuring only whole numbers and no negative values!
    2) Rounding the samples to integers.
    3) Ensuring no-negative numbers.
    4) Returns the generated, processed samples.
    """
    samples = np.random.normal(mean, sd, size)
    
    # Here we rounds the numbers to the nearest whole number
    samples = np.rint(samples).astype(int)
    
    # Ensuring no negative values 
    samples = np.maximum(samples, 0)
    
    return samples

Now let's create the function, which will sample the varible costs from the **Triangular distribution**:

In [3]:
def sample_costs_triangular(min, mode, max, size):
    """
    1) Samples the variables costs from a triangular distribution.
    2) Returning the samples.
    """
    
    return np.random.triangular(min, mode, max, size)

Now, let's the define the function which will evaluate the base model with the sampled data:

In [4]:
def simulate_profit(customers, price, cost_per_order, fixed_cost):
    """
    1) Calculates the profit, by the formula:
        customers * (price - cost_per_order) - fixed_cost.
    2) Returns the profit.
    """
    
    # First we multiply the amount of customers with the prive per order
    # Second, we are calculating the variable costs per order for all of the customers
    # Then, extracting the costs from the revenue
    revenue = customers * price
    total_var_cost = customers * cost_per_order
    profit = revenue - (total_var_cost + fixed_cost)
    
    return profit

Now that we have defined the sampling functions, let's use them for running of the simulations and processing the result:

In [5]:
def run_monte_carlo_simulation(simulations_size, 
                               mean_customers, sd_customers, 
                               cost_min, cost_mode, cost_max, 
                               price, fixed_cost):
    """
    1) Performs a Monte Carlo Simulation
    2) Generates Random Samples
    3) Calculating the base model using the samples
    4) Retuns the caclulated result
    """
    
    N_samples = sample_customers_normal(mean_customers, sd_customers, simulations_size)
    Cv_samples = sample_costs_triangular(cost_min, cost_mode, cost_max, simulations_size)
    
    # Calculating the profit, by appling the generated samples on the base model
    profits = simulate_profit(N_samples, price, Cv_samples, fixed_cost)
    
    return profits

Now let's define the input data and use it to estimate the profits:

In [18]:
# Example input data:
mean_customers = 100     # the average amount of customers per day
sd_customers = 20        # the standard deviation
price_per_order = 5.0    # price per order (fixed)
fixed_cost_daily = 200   # fixed daily overhead
cost_min = 0.5           # min variable cost
cost_mode = 0.6          # most likely variable cost
cost_max = 0.8           # max variable cost

# Performing the simulations
simulations_size = 100000
profits = run_monte_carlo_simulation(simulations_size, 
                                     mean_customers, sd_customers,
                                     cost_min, cost_mode, cost_max,
                                     price_per_order, fixed_cost_daily)

# Analyzing the profits
average_profit = np.mean(profits) # The average mean (the expected value)
prob_loss = np.mean(profits < 0)  # The probabiity that the profit will be negative
prob_high = np.mean(profits >= 500)  # The probability that the profit will exceed 500

# Calculating some Value-at-Risk (VaR)
var_5 = np.percentile(profits, 5) # The 5 percent of the profits will have a worse value than this
var_1 = np.percentile(profits, 1) # Worst case scenarios - scenarions with probability one percent!

# Most likely profit is approcimatelly equal to:
print(f"Average mean: {average_profit:.2f}")

# We multiply these by 100, to see approximately the percentages, but they are actually very less smaller than the shown values!
print(f"Probability of loss: {prob_loss*100:.1f}%")
print(f"Probability to get profit higher than 500: {prob_high*100:.1f}%")

# Edge case scenarios:
print(f"Value at risk 5%: {var_5:.2f}")
print(f"Value at risk 1%: {var_1:.2f}")

Average mean: 236.16
Probability of loss: 0.3%
Probability to get profit higher than 500: 0.1%
Value at risk 5%: 91.93
Value at risk 1%: 31.76


We can see that, from this simple, general algorithm we got some very interesting and useful results.

#### Ok, now in order to continue with the development of this algorithm, we are going to separate it in another file, which will be containing all of the it's functions and business logic.My idea is that as we introduce new concepts and techniques, we are going to explain and develop them here, and after that integrate them into the algorithm.

#### This way we will seperate the explanations of the new concepts and techniques from the actual implementation of the algorithm.
